In [0]:
from pyspark.sql.functions import col, lower,initcap,month,year,when,regexp_replace
from pyspark.sql.types import DoubleType, IntegerType

In [0]:
# acessando a tabela bronze
df_bronze = spark.table("projeto_profissional_bigdata.datatran_2018.bronze_datatran2018")

In [0]:
#conferencia do schema
df_bronze.printSchema()


In [0]:
#visualizando tabela bronze
df_bronze.limit(10).toPandas()

In [0]:
#criando tabela silver
df_silver = df_bronze.select("id","data_inversa","municipio","uf","br","km","causa_acidente","tipo_acidente")

df_silver.show(5)

In [0]:
#removendo linhas vazias
df_silver = df_silver.dropna(
    subset = ["id","data_inversa","municipio","uf","br","km","causa_acidente","tipo_acidente"]
)
df_silver.limit(5).toPandas()

In [0]:

df_silver = df_silver.withColumnRenamed("data_inversa","data_acidente")


In [0]:

#padronizando os nomes das colunas
df_silver = df_silver .withColumn("municipio",initcap(col("municipio")))
df_silver = df_silver .withColumn("causa_acidente",lower(col("causa_acidente")))
df_silver = df_silver .withColumn("tipo_acidente",lower(col("tipo_acidente")))
df_silver.limit(5).toPandas()


In [0]:
#criando mes, ano acidente
df_silver = df_silver .withColumn("mes_acidente",
                                  month(col("data_acidente")))
df_silver = df_silver .withColumn("ano_acidente",
                                   year(col("data_acidente")))

df_silver.limit(5).toPandas()


In [0]:
## mudando os valores da coluna mes acidente para os nomes dos meses
df_silver = df_silver.withColumn(
"nome_mes",
when(col("mes_acidente") == 1, "Janeiro")
.when(col("mes_acidente") == 2, "Fevereiro")
.when(col("mes_acidente") == 3, "Março")
.when(col("mes_acidente") == 4, "Abril")
.when(col("mes_acidente") == 5, "Maio")
.when(col("mes_acidente") == 6, "Junho")
.when(col("mes_acidente") == 7, "Julho")
.when(col("mes_acidente") == 8, "Agosto")
.when(col("mes_acidente") == 9, "Setembro")
.when(col("mes_acidente") == 10, "Outubro")
.when(col("mes_acidente") == 11, "Novembro")
.otherwise("dezembro")
)

df_silver.limit(5).toPandas


In [0]:
##contando linhas da coluna br = "NA"

df_silver.filter(col("br") == "NA").count()

In [0]:
## Trocando o valor da coluna br = NA  por Nome
df_silver = df_silver.withColumn("br",
when(col("br") == "NA", None).otherwise(col("br"))
)

In [0]:
df_silver = df_silver.dropna(subset=["br"])

In [0]:

df_silver = df_silver.withColumn("br", col("br").cast(IntegerType()))
df_silver.show(5)

In [0]:
##Trocando virgula por ponto na coluna km
df_silver = df_silver.withColumn("km",regexp_replace(col("km"), ",", "."))
df_silver.show(5)


In [0]:
df_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("projeto_profissional_bigdata.datatran_2018.silver_datatran18")

In [0]:
df=spark.read.table("projeto_profissional_bigdata.datatran_2018.silver_datatran18")
df.limit(5).toPandas()

In [0]:
%sql
select * from projeto_profissional_bigdata.datatran_2018.silver_datatran18 limit (20);